# Jointure des 7 tables IMDb

**Objectif :** construire une seule table avec **une ligne par titre**, à partir des 7 fichiers IMDb.

## Les clés de jointure

| Clé | Identifie | Où on la trouve |
|---|---|---|
| `tconst` | une œuvre (`tt0111161`) | toutes les tables `title.*` — s'appelle `titleId` dans `title.akas` et `parentTconst` (la série) dans `title.episode` |
| `nconst` | une personne (`nm0000151`) | `name.basics`, `title.principals`, et en listes dans `title.crew` |

## L'ordre des étapes

1. **`title.basics`** : la table pivot, filtrée dès le départ (type, années)
2. **`title.ratings`** : la note de chaque titre
3. **`title.crew`** : réalisateurs et scénaristes (des listes de `nconst` à éclater)
4. **`title.principals`** : les acteurs
5. **`name.basics`** : les noms des personnes trouvées aux étapes 3 et 4
6. **`title.akas`** : le titre traduit (ex. titre français)
7. **`title.episode`** : le lien entre épisodes et séries

Puis vérifications et export.

**Principe général :** on part de la table pivot filtrée, et on ne lit des autres tables **que les lignes qui concernent nos titres**. Sinon, certains fichiers (4,5 Go pour `title.principals`) ne tiennent pas en mémoire.

👉 Exécute cellule par cellule et regarde le résultat de chaque `.head()` et `.shape`.

## 0. Paramètres

Tout ce que tu peux modifier est regroupé ici. Plus les filtres sont stricts, plus le notebook est rapide.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)

# Dossier des fichiers IMDb (chemin relatif à ce notebook)
DOSSIER = Path("../../Dataset/SQL")
# Fichier de sortie
SORTIE = Path("../../Dataset/IMDb/imdb_7_tables_joined.csv")

# Filtres appliqués à la table pivot
TYPES = ["movie"]      # ex. ["movie", "tvSeries", "tvEpisode"] pour voir l'étape 7 en action
ANNEE_MIN = 2000       # None pour ne pas filtrer
ANNEE_MAX = None

NB_ACTEURS = 5         # nombre d'acteurs gardés par titre
REGION_TITRE = "FR"    # région du titre traduit (title.akas)

# Les gros fichiers sont lus par morceaux de CHUNKSIZE lignes
CHUNKSIZE = 1_000_000

## Fonctions utilitaires

- **`lire(nom)`** lit `nom.csv` s'il existe, sinon `nom.tsv`. IMDb écrit les valeurs manquantes `\N` : on les transforme en `NaN`. Tout est lu en texte (`dtype=str`) pour éviter les erreurs de type, et on convertit les colonnes numériques nous-mêmes ensuite.
- **`lire_filtre(nom, filtre)`** lit un fichier **par morceaux** et ne garde que les lignes utiles. C'est ce qui permet de traiter des fichiers de plusieurs Go.
- **`noms_par_titre(df, colonne)`** transforme une table *longue* (une ligne par titre × personne) en une ligne par titre : `"Nom1, Nom2, Nom3"`.

In [ ]:
# "\N" = valeur manquante chez IMDb, "" = cellule vide dans un CSV converti.
# keep_default_na=False évite que des titres comme "NA" ou "None" deviennent NaN.
VALEURS_MANQUANTES = ["\\N", ""]


def lire(nom, **kwargs):
    options = dict(na_values=VALEURS_MANQUANTES, keep_default_na=False, dtype=str, **kwargs)
    chemin_csv = DOSSIER / f"{nom}.csv"
    if chemin_csv.exists():
        return pd.read_csv(chemin_csv, **options)
    # quoting=3 : les guillemets dans les titres ne sont pas des délimiteurs dans les TSV IMDb
    return pd.read_csv(DOSSIER / f"{nom}.tsv", sep="\t", quoting=3, **options)


def lire_filtre(nom, filtre, usecols=None):
    """Lit un gros fichier par morceaux et ne garde que les lignes où filtre(morceau) est vrai."""
    morceaux = []
    for i, morceau in enumerate(lire(nom, usecols=usecols, chunksize=CHUNKSIZE)):
        morceaux.append(morceau[filtre(morceau)])
        print(f"\r{nom} : {(i + 1) * CHUNKSIZE:,} lignes parcourues", end="")
    print()
    return pd.concat(morceaux, ignore_index=True)


def noms_par_titre(df, nom_colonne, n=None):
    """Regroupe une table longue (tconst, primaryName) en une chaîne \"Nom1, Nom2\" par titre."""
    def joindre(noms):
        noms = noms.dropna()
        return ", ".join(noms.head(n) if n else noms)

    return (
        df.groupby("tconst", sort=False)["primaryName"]
        .agg(joindre)
        .rename(nom_colonne)
        .reset_index()
    )

## 1. `title.basics` : la table pivot

Une ligne = un titre (film, série, épisode, court-métrage…). C'est la base sur laquelle on va **ajouter des colonnes**, étape par étape.

On la filtre **tout de suite** : l'ensemble `ids` des `tconst` retenus servira ensuite à filtrer toutes les autres tables.

In [ ]:
def filtre_basics(morceau):
    annee = pd.to_numeric(morceau["startYear"], errors="coerce")
    garder = morceau["titleType"].isin(TYPES)
    if ANNEE_MIN is not None:
        garder &= annee >= ANNEE_MIN
    if ANNEE_MAX is not None:
        garder &= annee <= ANNEE_MAX
    return garder


titres = lire_filtre(
    "title.basics",
    filtre_basics,
    usecols=["tconst", "titleType", "primaryTitle", "originalTitle", "isAdult",
             "startYear", "endYear", "runtimeMinutes", "genres"],
)
for col in ["isAdult", "startYear", "endYear", "runtimeMinutes"]:
    titres[col] = pd.to_numeric(titres[col], errors="coerce").astype("Int64")

# Les identifiants retenus : le filtre de toutes les étapes suivantes
ids = set(titres["tconst"])

print(titres.shape)
titres.head()

## 2. `title.ratings` : les notes

Relation **1-1** : chaque titre a au plus une note.

- `how="left"` (LEFT JOIN) garde **tous** les titres, même ceux sans note : leur note vaut `NaN`.
- `validate="one_to_one"` fait planter le merge si une clé est en double des deux côtés. C'est une sécurité gratuite : utilise-la à chaque jointure.

Le fichier est petit (30 Mo) : pas besoin de le lire par morceaux.

In [ ]:
notes = lire("title.ratings")
notes["averageRating"] = pd.to_numeric(notes["averageRating"])
notes["numVotes"] = pd.to_numeric(notes["numVotes"]).astype("Int64")

df = titres.merge(notes, on="tconst", how="left", validate="one_to_one")

print(df.shape)  # même nombre de lignes que `titres` : un LEFT JOIN 1-1 n'en ajoute pas
print("Titres sans note :", df["averageRating"].isna().sum())
df.head()

## 3. `title.crew` : réalisateurs et scénaristes

Les colonnes `directors` et `writers` contiennent des **listes** de `nconst` séparées par des virgules : `nm0001,nm0002`.

Pour pouvoir les joindre aux noms, on les **éclate** : une ligne par couple (titre, personne). On obtient une table *longue* `equipe`. On n'y met pas encore les noms : ils arrivent à l'étape 5.

In [ ]:
crew = lire_filtre("title.crew", lambda d: d["tconst"].isin(ids))
crew.head()

In [ ]:
def eclater(colonne, role):
    longue = crew[["tconst", colonne]].dropna().copy()
    longue[colonne] = longue[colonne].str.split(",")   # "nm1,nm2" -> ["nm1", "nm2"]
    longue = longue.explode(colonne)                  # une ligne par élément de la liste
    longue = longue.rename(columns={colonne: "nconst"})
    longue["role"] = role
    return longue


equipe = pd.concat(
    [eclater("directors", "realisateur"), eclater("writers", "scenariste")],
    ignore_index=True,
)

print(equipe.shape)  # plus de lignes que `crew` : c'est normal, c'est l'effet de explode
equipe.head(10)

## 4. `title.principals` : les acteurs

C'est le plus gros fichier (4,5 Go) : une ligne par (titre, personne, métier). On garde seulement nos titres et les catégories `actor` / `actress`.

La colonne `ordering` donne l'ordre au générique : on trie dessus pour que les premiers acteurs soient les têtes d'affiche.

⏳ Cette cellule prend plusieurs minutes.

In [ ]:
principals = lire_filtre(
    "title.principals",
    lambda d: d["tconst"].isin(ids) & d["category"].isin(["actor", "actress"]),
    usecols=["tconst", "ordering", "nconst", "category", "characters"],
)
principals["ordering"] = pd.to_numeric(principals["ordering"])
principals = principals.sort_values(["tconst", "ordering"])

print(principals.shape)
principals.head(10)

## 5. `name.basics` : les noms des personnes

Pourquoi seulement maintenant ? Parce qu'on connaît enfin **tous les `nconst` dont on a besoin** (étapes 3 et 4). On peut donc ne lire que ces personnes-là, au lieu des 14 millions du fichier.

Ensuite :
1. on ajoute les noms aux tables longues `equipe` et `principals` (jointure sur `nconst`) ;
2. on les regroupe en **une ligne par titre** avec `noms_par_titre` ;
3. on joint ces résultats à `df` (jointure sur `tconst`).

In [ ]:
nconst_utiles = set(equipe["nconst"]) | set(principals["nconst"])
print(f"{len(nconst_utiles):,} personnes à retrouver")

personnes = lire_filtre(
    "name.basics",
    lambda d: d["nconst"].isin(nconst_utiles),
    usecols=["nconst", "primaryName"],
)
personnes.head()

In [ ]:
# 1. Ajouter les noms aux tables longues (jointure sur nconst)
equipe = equipe.merge(personnes, on="nconst", how="left")
principals = principals.merge(personnes, on="nconst", how="left")

# 2. Regrouper en une ligne par titre
realisateurs = noms_par_titre(equipe[equipe["role"] == "realisateur"], "realisateurs")
scenaristes = noms_par_titre(equipe[equipe["role"] == "scenariste"], "scenaristes")
acteurs = noms_par_titre(principals, "acteurs", n=NB_ACTEURS)

# 3. Joindre à la table principale (jointure sur tconst)
for table in [realisateurs, scenaristes, acteurs]:
    df = df.merge(table, on="tconst", how="left", validate="one_to_one")

print(df.shape)
df[["tconst", "primaryTitle", "realisateurs", "scenaristes", "acteurs"]].head()

## 6. `title.akas` : le titre traduit

Attention, ici la clé s'appelle **`titleId`** (et non `tconst`) : on la renomme avant la jointure.

Un même film peut avoir **plusieurs titres** pour une même région (titre d'exploitation, titre TV, titre de travail…). Si on joignait tel quel, le film serait dupliqué. On en garde donc un seul : celui marqué `imdbDisplay` s'il existe, sinon le premier.

⏳ Fichier de 3 Go : cette cellule prend quelques minutes.

In [ ]:
colonne_titre = f"titre_{REGION_TITRE.lower()}"

akas = lire_filtre(
    "title.akas",
    lambda d: d["titleId"].isin(ids) & (d["region"] == REGION_TITRE),
    usecols=["titleId", "ordering", "title", "region", "types"],
)
akas["ordering"] = pd.to_numeric(akas["ordering"])
akas["affichage"] = akas["types"].fillna("").str.contains("imdbDisplay")

titres_region = (
    akas.sort_values(["titleId", "affichage", "ordering"], ascending=[True, False, True])
    .drop_duplicates("titleId")  # un seul titre par film
    .rename(columns={"titleId": "tconst", "title": colonne_titre})
    [["tconst", colonne_titre]]
)

df = df.merge(titres_region, on="tconst", how="left", validate="one_to_one")

print(df.shape)
df[["tconst", "primaryTitle", colonne_titre]].head()

## 7. `title.episode` : épisodes et séries

Chaque ligne relie un **épisode** (`tconst`) à sa **série** (`parentTconst`). Cette table sert de deux façons :

- **a)** si notre titre est une **série**, il apparaît dans `parentTconst` : on compte ses épisodes et ses saisons ;
- **b)** si notre titre est un **épisode**, il apparaît dans `tconst` : on récupère sa série, sa saison et son numéro.

ℹ️ Avec `TYPES = ["movie"]`, ces colonnes seront vides : c'est normal, un film n'est ni une série ni un épisode. Ajoute `"tvSeries"` ou `"tvEpisode"` dans `TYPES` pour les voir se remplir.

In [ ]:
episodes = lire_filtre(
    "title.episode",
    lambda d: d["tconst"].isin(ids) | d["parentTconst"].isin(ids),
)
episodes["seasonNumber"] = pd.to_numeric(episodes["seasonNumber"], errors="coerce").astype("Int64")
episodes["episodeNumber"] = pd.to_numeric(episodes["episodeNumber"], errors="coerce").astype("Int64")

# a) Le titre est une série : combien d'épisodes et de saisons ?
stats_series = (
    episodes[episodes["parentTconst"].isin(ids)]
    .groupby("parentTconst")
    .agg(nb_episodes=("tconst", "count"), nb_saisons=("seasonNumber", "max"))
    .reset_index()
    .rename(columns={"parentTconst": "tconst"})
)

# b) Le titre est un épisode : de quelle série, quelle saison, quel numéro ?
infos_episodes = episodes.loc[
    episodes["tconst"].isin(ids), ["tconst", "parentTconst", "seasonNumber", "episodeNumber"]
]

df = df.merge(stats_series, on="tconst", how="left", validate="one_to_one")
df = df.merge(infos_episodes, on="tconst", how="left", validate="one_to_one")

print(df.shape)
print("Séries avec épisodes :", df["nb_episodes"].notna().sum())
print("Épisodes rattachés à une série :", df["parentTconst"].notna().sum())

## Vérifications et export

Une jointure ratée se voit généralement à deux endroits :
- des **doublons** de `tconst` : une table n'était pas 1-1 ;
- des colonnes **presque entièrement vides** : la clé ne correspondait pas (mauvais nom de colonne, mauvais type…).

In [ ]:
assert df["tconst"].is_unique, "Des titres sont dupliqués : une jointure n'est pas 1-1"

print(df.shape)
(df.isna().mean() * 100).round(1).sort_values(ascending=False).rename("% manquant")

In [ ]:
SORTIE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(SORTIE, index=False)
print(f"Fichier écrit : {SORTIE} ({len(df):,} lignes, {df.shape[1]} colonnes)")